# 04 — Comparaison de modèles avancés

Nous comparons la régression linéaire du notebook précédent à deux modèles d'ensemble pour prédire la cible numérique `Sales` : Random Forest et Gradient Boosting.

Le même découpage entraînement/test et les mêmes variables préparées sont réutilisés. Les recherches d'hyperparamètres utilisent uniquement des validations croisées sur l'entraînement ; le jeu de test reste réservé à l'évaluation finale. Le score présenté dans le tableau est le coefficient de détermination **R²** (plus élevé est meilleur).

## 1. Charger les jeux préparés et le baseline

Exécuter ce notebook depuis son dossier `notebooks/`. Les fichiers préparés contiennent déjà les variables numériques encodées et la colonne `Sales`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

DATA_DIR = Path("../data")
TARGET = "Sales"
train_data = pd.read_csv(DATA_DIR / "engineered_train.csv")
test_data = pd.read_csv(DATA_DIR / "engineered_test.csv")

for split_name, split_data in (("entraînement", train_data), ("test", test_data)):
    if TARGET not in split_data.columns:
        raise ValueError(f"La cible {TARGET!r} est absente du jeu de {split_name}.")
if list(train_data.columns) != list(test_data.columns):
    raise ValueError("Les colonnes des jeux d'entraînement et de test ne correspondent pas.")

X_train = train_data.drop(columns=TARGET)
y_train = train_data[TARGET]
X_test = test_data.drop(columns=TARGET)
y_test = test_data[TARGET]

if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train.dtypes):
    raise TypeError("Toutes les variables explicatives doivent être numériques.")
if X_train.isna().any().any() or X_test.isna().any().any() or y_train.isna().any() or y_test.isna().any():
    raise ValueError("Des valeurs manquantes ont été détectées dans les données.")
if not np.isfinite(X_train.to_numpy()).all() or not np.isfinite(X_test.to_numpy()).all():
    raise ValueError("Des valeurs infinies ont été détectées dans les variables explicatives.")
if not np.isfinite(y_train.to_numpy()).all() or not np.isfinite(y_test.to_numpy()).all():
    raise ValueError("Des valeurs infinies ont été détectées dans la cible.")

print(f"Entraînement : {X_train.shape[0]} lignes, {X_train.shape[1]} variables")
print(f"Test : {X_test.shape[0]} lignes, {X_test.shape[1]} variables")

Entraînement : 7840 lignes, 1197 variables
Test : 1960 lignes, 1197 variables


## 2. Entraîner le baseline

On réentraîne la régression linéaire sur le même jeu d'entraînement afin que les trois modèles soient comparés dans les mêmes conditions. Le score R² est calculé sur l'entraînement et sur le test.

In [2]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)
baseline_train_score = baseline_model.score(X_train, y_train)
baseline_test_prediction = baseline_model.predict(X_test)
baseline_test_score = r2_score(y_test, baseline_test_prediction)

print(f"Baseline — R² entraînement : {baseline_train_score:.4f}")
print(f"Baseline — R² test : {baseline_test_score:.4f}")
print(f"Baseline — MSE test : {mean_squared_error(y_test, baseline_test_prediction):,.2f}")

Baseline — R² entraînement : 0.2564
Baseline — R² test : 0.1638
Baseline — MSE test : 558,897.44


## 3. Recherche des hyperparamètres

`GridSearchCV` sélectionne les paramètres avec une validation croisée à 3 plis et le R² comme métrique. Les profondeurs et nombres d'estimateurs sont explorés pour chaque modèle. `max_features="sqrt"` limite le nombre de variables examinées par arbre, ce qui rend la recherche plus adaptée aux nombreuses colonnes encodées.

In [3]:
cv = 3

random_forest_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid={
        "n_estimators": [50, 100],
        "max_depth": [5, 10],
        "max_features": ["sqrt"],
    },
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
random_forest_search.fit(X_train, y_train)

gradient_boosting_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid={
        "n_estimators": [50, 100],
        "max_depth": [2, 3],
        "max_features": ["sqrt"],
    },
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True,
)
gradient_boosting_search.fit(X_train, y_train)

print(f"Random Forest — meilleur R² CV : {random_forest_search.best_score_:.4f}")
print(f"Random Forest — meilleurs paramètres : {random_forest_search.best_params_}")
print(f"Gradient Boosting — meilleur R² CV : {gradient_boosting_search.best_score_:.4f}")
print(f"Gradient Boosting — meilleurs paramètres : {gradient_boosting_search.best_params_}")

Random Forest — meilleur R² CV : 0.0952
Random Forest — meilleurs paramètres : {'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 100}
Gradient Boosting — meilleur R² CV : 0.1339
Gradient Boosting — meilleurs paramètres : {'max_depth': 2, 'max_features': 'sqrt', 'n_estimators': 100}


## 4. Évaluer sur les jeux d'entraînement et de test

Chaque `GridSearchCV` réentraîne automatiquement son meilleur estimateur sur l'ensemble de l'entraînement (`refit=True`). Nous évaluons ensuite ces modèles finaux sur le test, qui n'a pas participé à la recherche. Le R² d'entraînement aide à repérer un écart important avec le score de test, potentiellement signe de surapprentissage.

In [4]:
random_forest_model = random_forest_search.best_estimator_
gradient_boosting_model = gradient_boosting_search.best_estimator_

random_forest_train_score = random_forest_model.score(X_train, y_train)
random_forest_test_prediction = random_forest_model.predict(X_test)
random_forest_test_score = r2_score(y_test, random_forest_test_prediction)

gradient_boosting_train_score = gradient_boosting_model.score(X_train, y_train)
gradient_boosting_test_prediction = gradient_boosting_model.predict(X_test)
gradient_boosting_test_score = r2_score(y_test, gradient_boosting_test_prediction)

comparison = pd.DataFrame(
    [
        {
            "Model": "Baseline",
            "Train Score": baseline_train_score,
            "Test Score": baseline_test_score,
            "Best Params": "N/A",
        },
        {
            "Model": "Random Forest",
            "Train Score": random_forest_train_score,
            "Test Score": random_forest_test_score,
            "Best Params": random_forest_search.best_params_,
        },
        {
            "Model": "Gradient Boosting",
            "Train Score": gradient_boosting_train_score,
            "Test Score": gradient_boosting_test_score,
            "Best Params": gradient_boosting_search.best_params_,
        },
    ],
    columns=["Model", "Train Score", "Test Score", "Best Params"],
)
display(comparison)

,Model,Train Score,Test Score,Best Params
0,Baseline,0.256413,0.163828,N/A
1,Random Forest,0.256627,0.060745,"{'max_depth': 10, 'max_features': 'sqrt', 'n_e..."
2,Gradient Boosting,0.222790,0.124814,"{'max_depth': 2, 'max_features': 'sqrt', 'n_es..."


## 5. Conclusion

Le choix du modèle se fait d'abord sur le meilleur R² moyen en validation croisée, calculé uniquement à partir de l'entraînement ; le score de test reste une vérification finale de sa capacité à généraliser. La comparaison ci-dessus inclut aussi le baseline. Un écart élevé entre les scores d'entraînement et de test peut signaler du surapprentissage. Le texte généré ci-dessous indique le meilleur des modèles avancés selon la validation croisée et s'il dépasse le baseline sur le test.

In [5]:
advanced_scores = {
    "Random Forest": random_forest_search.best_score_,
    "Gradient Boosting": gradient_boosting_search.best_score_,
}
best_advanced_name = max(advanced_scores, key=advanced_scores.get)
best_advanced_test_score = comparison.loc[
    comparison["Model"] == best_advanced_name, "Test Score"
].iloc[0]

print(f"Meilleur modèle avancé selon le R² moyen en validation croisée : {best_advanced_name}.")
if best_advanced_test_score > baseline_test_score:
    print("Conclusion : ce modèle avancé dépasse le baseline sur le jeu de test ; il est le meilleur choix parmi les modèles comparés, sous réserve de confirmer ce résultat sur de nouvelles données.")
else:
    print("Conclusion : le modèle avancé sélectionné par validation croisée ne dépasse pas le baseline sur le jeu de test ; la régression linéaire reste préférable pour l'instant.")

Meilleur modèle avancé selon le R² moyen en validation croisée : Gradient Boosting.
Conclusion : le modèle avancé sélectionné par validation croisée ne dépasse pas le baseline sur le jeu de test ; la régression linéaire reste préférable pour l'instant.


## 6. Enregistrer le modèle retenu

La régression linéaire obtient le meilleur R² sur le jeu de test parmi les modèles comparés. Cette cellule enregistre le modèle entraîné pour l'application Streamlit.

In [ ]:
import joblib

model_output = DATA_DIR / "sales_model.joblib"
joblib.dump(baseline_model, model_output)
print(f"Modèle enregistré : {model_output}")